# Malware Detection Using Deep Learning and Correlation-Based Feature Selection
## Paper replication notebook

**Paper:** Alomari et al. (2023), *Malware Detection Using Deep Learning and Correlation-Based Feature Selection*, Symmetry 15(1), 123.

This notebook is designed around the uploaded `Malware dataset.csv`.

### Important reproducibility notes
- The uploaded file contains 100,000 rows and 35 columns, matching the paper's first dataset.
- `hash` is an identifier and is excluded from model features.
- `classification` is encoded as benign=0 and malware=1.
- The paper specifies 80/20 train/test, 20% of training as validation, batch size 100, Adam, 20 epochs, sparse categorical cross-entropy, and a dense model with six 50-unit ReLU Dense layers followed by a 2-unit softmax output.
- The paper does not clearly document a feature scaler or a random seed. This notebook uses a **training-only StandardScaler** for numerical stability and `random_state=42` for reproducibility. This is an implementation choice and must be stated when comparing against the paper.
- The primary experiments use the paper's predefined 33/27/24/21/14 feature groups. An optional hash-grouped split is included separately to investigate possible group leakage.


In [ ]:
# 1. Import libraries
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)


In [ ]:
# 2. Load dataset
CSV_PATH = "Malware dataset.csv"  # change if your file has another path
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
display(df.head())


In [ ]:
# 3. Dataset inspection
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
for c in df.columns:
    print(c)

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing").query("missing > 0"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nClass distribution:")
display(df["classification"].value_counts(dropna=False))

print("\nUnique hashes:", df["hash"].nunique())


In [ ]:
# 4. Data cleaning

# Paper preprocessing says to check special/missing values, encode the target,
# and drop hash. The uploaded dataset has no missing values.
print("Missing values before cleaning:", df.isna().sum().sum())

# Encode target
df["classification"] = df["classification"].map({"benign": 0, "malware": 1})

if df["classification"].isna().any():
    raise ValueError("Unexpected target labels found.")

# Drop identifier/hash from predictive features
X = df.drop(columns=["hash", "classification"]).copy()
y = df["classification"].astype(int).to_numpy()

# Report constant features but retain them for the primary paper-replication
# experiment because the paper's feature counts are based on 33 predictors.
constant_features = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]

print("Constant features:")
print(constant_features)
print("Number of predictors after dropping hash:", X.shape[1])


In [ ]:
# 5. Exploratory analysis
display(X.describe().T)

plt.figure(figsize=(7,4))
sns.countplot(x=df["classification"].map({0:"benign",1:"malware"}))
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


In [ ]:
# 6. Correlation-based feature selection

# Pearson correlation is appropriate here because the predictors are numerical
# and the target is binary (the Pearson binary-target correlation is the
# point-biserial correlation).

# IMPORTANT: for leakage-controlled feature selection, compute this on the
# training portion only. The paper's predefined feature groups are reproduced
# below exactly.

paper_corr = X.join(df["classification"]).corr(numeric_only=True)["classification"]              .drop("classification").sort_values(key=lambda s: s.abs(), ascending=False)

correlation_table = paper_corr.rename("Correlation with target").reset_index()
correlation_table.columns = ["Feature", "Correlation with target"]

# Paper's four cumulative drop groups
drop_1 = [
    "vm_truncate_count","shared_vm","exec_vm","nvcsw","maj_flt","utime"
]
drop_2 = drop_1 + ["static_prio","map_count","end_data"]
drop_3 = drop_2 + ["nivcsw","fs_excl_counter","reserved_vm"]
drop_4 = drop_3 + [
    "mm_users","state","total_vm","free_area_cache","stime","gtime","millisecond"
]

all_features = list(X.columns)
feature_sets = {
    "33": all_features,
    "27": [c for c in all_features if c not in drop_1],
    "24": [c for c in all_features if c not in drop_2],
    "21": [c for c in all_features if c not in drop_3],
    "14": [c for c in all_features if c not in drop_4],
}

print("Feature counts:", {k: len(v) for k,v in feature_sets.items()})
display(correlation_table)


In [ ]:
# Feature selection decision table
decision = correlation_table.copy()
decision["Selected_33"] = decision["Feature"].isin(feature_sets["33"])
decision["Selected_27"] = decision["Feature"].isin(feature_sets["27"])
decision["Selected_24"] = decision["Feature"].isin(feature_sets["24"])
decision["Selected_21"] = decision["Feature"].isin(feature_sets["21"])
decision["Selected_14"] = decision["Feature"].isin(feature_sets["14"])
display(decision)


In [ ]:
# 7. Train/test split

# The paper evaluates an 80/20 split. Stratification preserves the balanced
# malware/benign class distribution.
train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

# 20% of the training set becomes validation data, matching the paper.
train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.20,
    stratify=y[train_idx],
    random_state=SEED
)

print("Training:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))


In [ ]:
# 8. Feature scaling / encoding

# No categorical predictors are present after dropping hash and the target.
# StandardScaler is fit ONLY on the training data to prevent test leakage.
#
# This scaler is an explicit reproducibility/stability choice because the paper
# does not clearly specify a scaling procedure.

def prepare_arrays(features):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(df.iloc[train_idx][features]).astype(np.float32)
    X_val   = scaler.transform(df.iloc[val_idx][features]).astype(np.float32)
    X_test  = scaler.transform(df.iloc[test_idx][features]).astype(np.float32)

    y_train = y[train_idx]
    y_val   = y[val_idx]
    y_test  = y[test_idx]

    return X_train, X_val, X_test, y_train, y_val, y_test, scaler


In [ ]:
# 9. Build model

class PaperDNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        # Paper description:
        # first Dense layer + five further hidden Dense layers, 50 ReLU units each
        # followed by a two-output softmax layer.
        layers = [
            nn.Linear(n_features, 50),
            nn.ReLU(),
        ]

        for _ in range(5):
            layers.extend([
                nn.Linear(50, 50),
                nn.ReLU()
            ])

        layers.append(nn.Linear(50, 2))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

def build_model(n_features):
    return PaperDNN(n_features)

model = build_model(len(feature_sets["33"]))
print(model)


In [ ]:
# 10. Train model

def train_model(features, epochs=20, batch_size=100, lr=0.001):
    X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_arrays(features)

    model = build_model(len(features))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_ds = TensorDataset(
        torch.tensor(X_train),
        torch.tensor(y_train, dtype=torch.long)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(yb)

        model.eval()
        with torch.no_grad():
            val_logits = model(torch.tensor(X_val))
            val_loss = criterion(
                val_logits,
                torch.tensor(y_val, dtype=torch.long)
            ).item()
            val_pred = val_logits.argmax(dim=1).numpy()

        history["loss"].append(running_loss / total)
        history["accuracy"].append(correct / total)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(accuracy_score(y_val, val_pred))

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"loss={history['loss'][-1]:.4f} | "
            f"acc={history['accuracy'][-1]:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={history['val_accuracy'][-1]:.4f}"
        )

    return model, scaler, history, (X_test, y_test)


In [ ]:
# Run one scenario
model, scaler, history, test_data = train_model(feature_sets["33"])


In [ ]:
# 11. Evaluate model

def evaluate_model(model, scaler, features, test_data):
    X_test, y_test = test_data

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_test))
        probabilities = torch.softmax(logits, dim=1)[:,1].numpy()
        predictions = logits.argmax(dim=1).numpy()

    metrics = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    }

    print(pd.DataFrame(metrics.items(), columns=["Metric","Score"]))
    print("\nClassification report:")
    print(classification_report(y_test, predictions, target_names=["benign","malware"]))

    return predictions, probabilities, metrics

predictions, probabilities, metrics = evaluate_model(
    model, scaler, feature_sets["33"], test_data
)


In [ ]:
# 12. Confusion matrix
cm = confusion_matrix(test_data[1], predictions)

plt.figure(figsize=(5,4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["benign","malware"],
    yticklabels=["benign","malware"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# 13. Training graphs

plt.figure(figsize=(7,4))
plt.plot(history["accuracy"], label="Training accuracy")
plt.plot(history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(7,4))
plt.plot(history["loss"], label="Training loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

fpr, tpr, _ = roc_curve(test_data[1], probabilities)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(test_data[1], probabilities):.4f}")
plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


In [ ]:
# 14. Save model

os.makedirs("saved_models", exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "features": feature_sets["33"],
        "seed": SEED,
        "architecture": "6 Dense(50, ReLU) layers + Dense(2, Softmax-equivalent logits)"
    },
    "saved_models/malware_dnn_33_features.pt"
)

# Save the scaler parameters needed for inference
np.save("saved_models/malware_dnn_33_mean.npy", scaler.mean_)
np.save("saved_models/malware_dnn_33_scale.npy", scaler.scale_)

print("Model and preprocessing parameters saved.")


In [ ]:
# 15. Run all five paper feature-selection scenarios

all_results = []

for scenario, features in feature_sets.items():
    print(f"\n===== Scenario: {scenario} features =====")
    model, scaler, history, test_data = train_model(
        features,
        epochs=20,
        batch_size=100,
        lr=0.001
    )

    pred, prob, metrics = evaluate_model(model, scaler, features, test_data)

    row = {
        "Scenario": f"{scenario} features",
        "Features": len(features),
        **metrics
    }
    all_results.append(row)

results_df = pd.DataFrame(all_results)
display(results_df)
results_df.to_csv("malware_dnn_results.csv", index=False)


In [ ]:
# 16. Comparison with paper

paper_table = pd.DataFrame({
    "Scenario": ["33 features","27 features","24 features","21 features","14 features"],
    "Paper validation accuracy (%)": [99.99,99.92,99.95,99.75,94.15],
    "Paper test accuracy (%)": [100,99.54,99.91,99.79,93.79],
    "Paper precision (%)": [100,100,99.9,99.8,93.9],
    "Paper recall (%)": [100,100,99.9,99.8,93.8],
    "Paper F1 (%)": [100,100,99.9,99.8,93.8],
})

display(paper_table)

# Merge your measured results with the paper results
comparison = paper_table.merge(results_df, on="Scenario", how="left")
display(comparison)

print("""
Interpretation:
1. Paper values are published values and are never substituted for our measured results.
2. Our implementation uses StandardScaler fit only on training data and seed=42;
   the paper does not clearly document those choices.
3. Our ROC-AUC is an additional metric; the paper reports accuracy, precision,
   recall, and F1-score rather than ROC-AUC for these scenarios.
4. Small differences are therefore expected even when the dataset and architecture match.
""")


In [ ]:
# Optional leakage-controlled analysis: split by hash

# This is NOT the primary paper replication. It checks whether the same hash/sample
# group can cross train and test boundaries.

groups = df["hash"].to_numpy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
group_train, group_test = next(gss.split(X, y, groups=groups))

print("Group-aware train rows:", len(group_train))
print("Group-aware test rows:", len(group_test))
print("Train hashes:", df.iloc[group_train]["hash"].nunique())
print("Test hashes:", df.iloc[group_test]["hash"].nunique())
print("Hash overlap:", len(
    set(df.iloc[group_train]["hash"]) & set(df.iloc[group_test]["hash"])
))

# Use this experiment separately in the report. Do not mix its scores with the
# paper-replication scores.


## Academic conclusion

The uploaded CSV matches the paper's first dataset in size, class balance, and feature structure. The implementation reproduces the paper's five dense-model feature scenarios and training configuration as closely as the paper documents.

The most important reproducibility limitation is that the paper does not clearly specify the random seed, exact software versions, or a feature-scaling procedure. The paper also does not document a hash-grouped split, although the uploaded dataset contains repeated hashes. Therefore, differences between our measured results and the published results should not be interpreted as an error without considering these implementation details.
